In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [0]:
source_table = "retail_project.silver.sales_cleaned"
target_table = "retail_project.gold.dim_products"
silver_df = spark.table(source_table)


In [0]:
new_product_data = (
    silver_df.select("product_name", "category").distinct()
    .withColumn("product_key", F.sha2(F.col("product_name"), 256))
)

In [0]:
def upsert_product_dimension(df, target_table):
    if not spark.catalog.tableExists(target_table):
        df.write.format("delta").saveAsTable(target_table)
    else:
        target_data = DeltaTable.forName(spark, target_table)

        target_data.alias("t").merge(
            df.alias("s"), 
            "t.product_name = s.product_name"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


In [0]:
upsert_product_dimension(new_product_data, target_table)